# 02 — Data cleaning and quality validation

This notebook combines the six raw city files into one tidy daily dataset. It standardizes data types and validates completeness, uniqueness and basic physical consistency before saving any processed data.

No observation is silently removed or corrected. Any problem is included in the quality report and stops the processed dataset from being saved.

## 1. Project setup

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "scripts").is_dir():
    raise RuntimeError(
        "Could not locate the project root. Start Jupyter from the "
        "Weather-Report repository or its notebooks directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT.name}")

Project root: Weather-Report


In [3]:
from datetime import date

import pandas as pd

from scripts import (
    build_quality_report,
    clean_weather_data,
    discover_raw_csv_files,
    get_cities,
    load_raw_weather_csvs,
    save_processed_csv,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Cleaning configuration

The expected dates and cities must match the data-collection notebook. Keeping these assumptions explicit prevents a partial download from being mistaken for a complete dataset.

In [4]:
START_DATE = date(1940, 1, 1)
END_DATE = date(2025, 12, 31)
CITIES = get_cities()
CITY_SLUGS = [city.slug for city in CITIES]
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_FILE = PROCESSED_DATA_DIR / "weather_daily_1940_2025.csv"
ALLOWED_MISSING_VALUES = {
    "precipitation_sum": {date(1940, 1, 1)},
}

display_raw_path = Path(PROJECT_ROOT.name) / RAW_DATA_DIR.relative_to(PROJECT_ROOT)
display_processed_path = (
    Path(PROJECT_ROOT.name) / PROCESSED_FILE.relative_to(PROJECT_ROOT)
)

print(f"Expected cities: {', '.join(CITY_SLUGS)}")
print(f"Expected period: {START_DATE} to {END_DATE}")
print(f"Raw directory: {display_raw_path}")
print(f"Processed file: {display_processed_path}")

Expected cities: milan, turin, rome, bari, palermo, cagliari
Expected period: 1940-01-01 to 2025-12-31
Raw directory: Weather-Report/data/raw
Processed file: Weather-Report/data/processed/weather_daily_1940_2025.csv


## 3. Discover and load the raw files

Exactly one ERA5 CSV file must exist for each expected city. JSON responses remain preserved in the raw directory but are not needed for tabular cleaning.

In [5]:
raw_csv_files = discover_raw_csv_files(RAW_DATA_DIR, CITY_SLUGS)

for file_path in raw_csv_files:
    display_path = Path(PROJECT_ROOT.name) / file_path.relative_to(PROJECT_ROOT)
    print(display_path)

Weather-Report/data/raw/milan_19400101_20251231_era5.csv
Weather-Report/data/raw/turin_19400101_20251231_era5.csv
Weather-Report/data/raw/rome_19400101_20251231_era5.csv
Weather-Report/data/raw/bari_19400101_20251231_era5.csv
Weather-Report/data/raw/palermo_19400101_20251231_era5.csv
Weather-Report/data/raw/cagliari_19400101_20251231_era5.csv


In [6]:
raw_weather = load_raw_weather_csvs(raw_csv_files)

print(f"Raw rows: {len(raw_weather):,}")
print(f"Raw columns: {raw_weather.shape[1]}")
print(f"Memory usage: {raw_weather.memory_usage(deep=True).sum() / 1_000_000:.1f} MB")
raw_weather.head()

Raw rows: 188,472
Raw columns: 13
Memory usage: 60.6 MB


,city,city_slug,requested_latitude,requested_longitude,grid_latitude,grid_longitude,grid_elevation,time,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,source_file
0,Milan,milan,45.4642,9.19,45.5,9.25,147.0,1940-01-01,-1.8,2.3,-5.5,NaN,milan_19400101_20251231_era5.csv
1,Milan,milan,45.4642,9.19,45.5,9.25,147.0,1940-01-02,-2.1,1.4,-5.3,0.0,milan_19400101_20251231_era5.csv
2,Milan,milan,45.4642,9.19,45.5,9.25,147.0,1940-01-03,-4.2,-1.7,-6.0,0.0,milan_19400101_20251231_era5.csv
3,Milan,milan,45.4642,9.19,45.5,9.25,147.0,1940-01-04,0.1,3.2,-2.9,1.5,milan_19400101_20251231_era5.csv
4,Milan,milan,45.4642,9.19,45.5,9.25,147.0,1940-01-05,2.4,5.8,-1.0,0.3,milan_19400101_20251231_era5.csv


## 4. Standardize the dataset

The raw `time` field becomes `date`. Dates and numeric columns are parsed explicitly, city identifiers are standardized as strings, and rows are ordered by city and date. Conversion problems become missing values and are therefore visible in the quality report.

In [7]:
weather = clean_weather_data(raw_weather)

print(f"Cleaned shape: {weather.shape}")
display(weather.dtypes.to_frame(name="dtype"))
weather.head()

Cleaned shape: (188472, 13)


,dtype
city,string
city_slug,string
requested_latitude,float64
requested_longitude,float64
grid_latitude,float64
grid_longitude,float64
grid_elevation,float64
date,datetime64[us]
temperature_2m_mean,float64
temperature_2m_max,float64


,city,city_slug,requested_latitude,requested_longitude,grid_latitude,grid_longitude,grid_elevation,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,source_file
0,Bari,bari,41.1171,16.8719,41.0,16.75,9.0,1940-01-01,1.6,3.9,-1.6,NaN,bari_19400101_20251231_era5.csv
1,Bari,bari,41.1171,16.8719,41.0,16.75,9.0,1940-01-02,2.6,4.9,1.2,1.3,bari_19400101_20251231_era5.csv
2,Bari,bari,41.1171,16.8719,41.0,16.75,9.0,1940-01-03,1.5,4.4,-0.6,0.0,bari_19400101_20251231_era5.csv
3,Bari,bari,41.1171,16.8719,41.0,16.75,9.0,1940-01-04,6.6,9.0,2.3,7.6,bari_19400101_20251231_era5.csv
4,Bari,bari,41.1171,16.8719,41.0,16.75,9.0,1940-01-05,9.4,12.0,7.5,11.1,bari_19400101_20251231_era5.csv


## 5. Build the quality report

The report checks:

- one observation per city and date;
- every calendar day from 1940 through 2025;
- no unexpected missing analysis values;
- `temperature minimum <= temperature mean <= temperature maximum`;
- non-negative precipitation;
- no dates outside the requested period;
- no unexpected cities.

Open-Meteo returns missing precipitation for 1 January 1940 in every city. This is treated as a systematic boundary limitation observed at the beginning of the ERA5 archive. The six values remain missing, are explicitly reported as allowed, and will be excluded from precipitation calculations rather than imputed.

In [8]:
quality_report = build_quality_report(
    weather,
    expected_city_slugs=CITY_SLUGS,
    start_date=START_DATE,
    end_date=END_DATE,
    allowed_missing_values=ALLOWED_MISSING_VALUES,
)

display(quality_report.city_summary)
display(
    pd.Series(quality_report.issue_counts, name="count")
    .rename_axis("quality_check")
    .to_frame()
)
print("Allowed missing values:", quality_report.allowed_null_counts)
print(f"Dataset valid: {quality_report.is_valid}")

,city_slug,row_count,expected_rows,first_date,last_date,missing_dates,duplicate_rows,unexpected_missing_values,allowed_missing_values,is_complete
0,milan,31412,31412,1940-01-01,2025-12-31,0,0,0,1,True
1,turin,31412,31412,1940-01-01,2025-12-31,0,0,0,1,True
2,rome,31412,31412,1940-01-01,2025-12-31,0,0,0,1,True
3,bari,31412,31412,1940-01-01,2025-12-31,0,0,0,1,True
4,palermo,31412,31412,1940-01-01,2025-12-31,0,0,0,1,True
5,cagliari,31412,31412,1940-01-01,2025-12-31,0,0,0,1,True


,count
quality_check,
duplicate_rows,0
invalid_temperature_rows,0
negative_precipitation_rows,0
out_of_range_rows,0
missing_dates,0
unexpected_null_values,0
unexpected_cities,0


Allowed missing values: {'city': 0, 'city_slug': 0, 'date': 0, 'temperature_2m_mean': 0, 'temperature_2m_max': 0, 'temperature_2m_min': 0, 'precipitation_sum': 6}
Dataset valid: True


In [9]:
if not quality_report.is_valid:
    raise ValueError(
        "Mandatory quality checks failed. Inspect the issue counts and "
        "the detailed tables stored in quality_report before continuing."
    )

print("All mandatory quality checks passed.")

All mandatory quality checks passed.


### Inspecting issues if validation fails

The detailed tables are available without changing the underlying data:

```python
quality_report.duplicate_rows
quality_report.invalid_temperature_rows
quality_report.negative_precipitation_rows
quality_report.out_of_range_rows
quality_report.missing_dates
quality_report.null_counts
quality_report.allowed_null_counts
```

## 6. Save and verify the processed dataset

The output contains all validated observations and a `source_file` field linking every row to its raw CSV. Running this cell intentionally regenerates the processed file from the preserved raw inputs.

In [10]:
saved_path = save_processed_csv(weather, PROCESSED_FILE, overwrite=True)
display_saved_path = Path(PROJECT_ROOT.name) / saved_path.relative_to(PROJECT_ROOT)

print(f"Saved {len(weather):,} rows to {display_saved_path}")
print(f"File size: {saved_path.stat().st_size / 1_000_000:.1f} MB")

Saved 188,472 rows to Weather-Report/data/processed/weather_daily_1940_2025.csv
File size: 20.1 MB


In [11]:
verification_data = pd.read_csv(PROCESSED_FILE, parse_dates=["date"])

assert len(verification_data) == len(weather)
assert list(verification_data.columns) == list(weather.columns)
assert verification_data["date"].min().date() == START_DATE
assert verification_data["date"].max().date() == END_DATE
assert set(verification_data["city_slug"].unique()) == set(CITY_SLUGS)

print("Saved dataset reloaded successfully and passed integrity checks.")

Saved dataset reloaded successfully and passed integrity checks.


## Result and next step

The result is one validated daily dataset covering all six cities from 1940 through 2025. The next notebook will perform exploratory data analysis: distributions, monthly seasonality, city comparisons and initial checks for long-term patterns.